# UPI Data Cleaning
Cleans three raw NPCI datasets and exports analysis-ready CSVs.

| Part | Dataset | Output |
|------|---------|--------|
| 1 | Merchant category (MCC) data | `merchant_category_data.csv` |
| 2 | State-wise UPI data | `state_wise_upi.csv` |
| 3 | Monthly UPI statistics | `upi_df.csv` |

**Flow for every dataset:** Load & combine files → Standardise columns → Extract Month/Date → Financial Year → Clean numeric columns → Remove invalid rows → Validate → Export

## 0. Setup: imports, paths and reusable helpers

In [35]:
import os
import re
import glob
import numpy as np
import pandas as pd

# ---- Input folders ----
MERCHANT_FOLDER = r"C:\Users\nazmi\OneDrive\Desktop\UPI_data\merchant upi dataset\AII"
STATE_FOLDER    = r"C:\Users\nazmi\OneDrive\Desktop\UPI_data\state-wise"
MONTHLY_FOLDER  = r"C:\Users\nazmi\OneDrive\Desktop\UPI_data\monthly-statistics"

# ---- Output files ----
merchant_category_dataf = r"C:\Users\nazmi\Downloads\merchant_category_data.csv"
STATE_OUT    = r"C:\Users\nazmi\Downloads\state_wise_upi.csv"
MONTHLY_OUT  = r"C:\Users\nazmi\Downloads\upi_df.csv"

MONTHS = "Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec"

In [2]:
def to_numeric_clean(series):
    """Remove thousand-separator commas and convert to numeric (bad values -> NaN)."""
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )


def add_financial_year(df, date_col="Date"):
    """Add Indian Financial Year label (Apr-Mar), e.g. Apr-2023 -> 'FY 2023-24'."""
    year = df[date_col].dt.year
    start_year = year.where(df[date_col].dt.month >= 4, year - 1)   # Jan-Mar belongs to previous FY
    df["Financial_Year"] = (
        "FY " + start_year.astype("Int64").astype(str)
        + "-" + (start_year + 1).astype("Int64").astype(str).str[-2:]
    )
    return df

## 1. Merchant Category (MCC) Data
### 1.1 Load and combine all Excel files
Each file is read without header; only the first 5 columns are kept and the header row is dropped.

In [3]:
merchant_files = glob.glob(os.path.join(MERCHANT_FOLDER, "*.xlsx"))
print("Number of files:", len(merchant_files))

frames = []
for file in merchant_files:
    temp = pd.read_excel(file, header=None)      # read without header
    temp = temp.iloc[:, :5]                      # keep first 5 columns only
    temp = temp.iloc[1:].copy()                  # drop the header row
    temp.columns = ["Type", "MCC", "Description", "Volume_Mn", "Value_Cr"]   # fixed column names
    temp["Source_File"] = os.path.basename(file) # keep original filename
    frames.append(temp)

merch = pd.concat(frames, ignore_index=True)
print(merch.shape)

Number of files: 40
(1278, 6)


In [4]:
# Sanity check: rows read from each file
for file in merchant_files:
    print(os.path.basename(file), "->", pd.read_excel(file, header=None).shape)

dec 2024.xlsx -> (32, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Apr.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Aug.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Dec.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Jul.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Jun.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-May.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Nov.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Oct.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2023-Sep.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2024-Apr.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2024-Aug.xlsx -> (33, 5)
Ecosystem-Statistics-UPI-Merchant-category-classification-2024

### 1.2 Extract Month and Year from file name
Files follow two naming styles: `...-2023-Apr...` and `Apr 2023.xlsx`.

In [5]:
def extract_month_year(filename):
    """Return (Month, Year) from either supported file-name pattern."""
    m = re.search(rf"-(\d{{4}})-({MONTHS})", filename, flags=re.I)        # style 1: -2023-Apr
    if m:
        return m.group(2).capitalize(), m.group(1)
    m = re.search(rf"^({MONTHS})\s+(\d{{4}})\.xlsx$", filename, flags=re.I)  # style 2: Apr 2023.xlsx
    if m:
        return m.group(1).capitalize(), m.group(2)
    return None, None

lookup = {f: extract_month_year(f) for f in merch["Source_File"].unique()}
merch["Month"] = merch["Source_File"].map(lambda f: lookup[f][0])
merch["Year"]  = merch["Source_File"].map(lambda f: lookup[f][1])

# Every file must have been matched
print("Missing Month:", merch["Month"].isna().sum())
merch[["Source_File", "Month", "Year"]].drop_duplicates()

Missing Month: 0


,Source_File,Month,Year
0,dec 2024.xlsx,Dec,2024
31,Ecosystem-Statistics-UPI-Merchant-category-cla...,Apr,2023
63,Ecosystem-Statistics-UPI-Merchant-category-cla...,Aug,2023
95,Ecosystem-Statistics-UPI-Merchant-category-cla...,Dec,2023
127,Ecosystem-Statistics-UPI-Merchant-category-cla...,Jul,2023
159,Ecosystem-Statistics-UPI-Merchant-category-cla...,Jun,2023
191,Ecosystem-Statistics-UPI-Merchant-category-cla...,May,2023
223,Ecosystem-Statistics-UPI-Merchant-category-cla...,Nov,2023
255,Ecosystem-Statistics-UPI-Merchant-category-cla...,Oct,2023
287,Ecosystem-Statistics-UPI-Merchant-category-cla...,Sep,2023


### 1.3 Create Date and Financial Year

In [6]:
merch["Date"] = pd.to_datetime(merch["Month"] + "-" + merch["Year"], format="%b-%Y", errors="coerce")
merch["Year"] = pd.to_numeric(merch["Year"], errors="coerce")
merch["Month_Num"] = merch["Date"].dt.month
merch = add_financial_year(merch)

print(merch["Financial_Year"].value_counts().sort_index())

Financial_Year
FY 2023-24    384
FY 2024-25    382
FY 2025-26    384
FY 2026-27    128
Name: count, dtype: int64


### 1.4 Clean numeric columns
Commas are stripped **before** conversion, otherwise values like `1,234.5` would silently become NaN.

In [7]:
merch["Volume_Mn"] = to_numeric_clean(merch["Volume_Mn"])
merch["Value_Cr"]  = to_numeric_clean(merch["Value_Cr"])
merch.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1278 entries, 0 to 1277
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Type            188 non-null    object        
 1   MCC             1230 non-null   object        
 2   Description     1222 non-null   object        
 3   Volume_Mn       1240 non-null   float64       
 4   Value_Cr        1240 non-null   float64       
 5   Source_File     1278 non-null   object        
 6   Month           1278 non-null   object        
 7   Year            1278 non-null   int64         
 8   Date            1278 non-null   datetime64[ns]
 9   Month_Num       1278 non-null   int32         
 10  Financial_Year  1278 non-null   object        
dtypes: datetime64[ns](1), float64(2), int32(1), int64(1), object(6)
memory usage: 105.0+ KB


### 1.5 Remove invalid rows

In [8]:
# 'Type' is not needed
merch = merch.drop(columns=["Type"])

# Remove 'Total' summary rows (they would double count)
merch = merch[merch["Description"] != "Total"]

# Remove empty rows (no MCC) and repeated header rows inside the data
merch = merch.dropna(subset=["MCC"])
merch = merch[merch["MCC"].astype(str).str.lower() != "mcc"]

# Missing description -> 'Others'
merch["Description"] = merch["Description"].fillna("Others")

### 1.6 Validate

In [9]:
print("Shape:", merch.shape)
print("\nNull values:\n", merch.isnull().sum())
print("\nDuplicates (MCC + Date):", merch.duplicated(subset=["MCC", "Date"]).sum())
print("\nMonths per Financial Year:\n", merch.groupby("Financial_Year")["Month"].nunique())
print("\nRows per Financial Year:\n", merch["Financial_Year"].value_counts().sort_index())
merch.head()

Shape: (1193, 10)

Null values:
 MCC               0
Description       0
Volume_Mn         0
Value_Cr          0
Source_File       0
Month             0
Year              0
Date              0
Month_Num         0
Financial_Year    0
dtype: int64

Duplicates (MCC + Date): 0

Months per Financial Year:
 Financial_Year
FY 2023-24    12
FY 2024-25    12
FY 2025-26    12
FY 2026-27     4
Name: Month, dtype: int64

Rows per Financial Year:
 Financial_Year
FY 2023-24    360
FY 2024-25    360
FY 2025-26    357
FY 2026-27    116
Name: count, dtype: int64


,MCC,Description,Volume_Mn,Value_Cr,Source_File,Month,Year,Date,Month_Num,Financial_Year
0,5411,Groceries and supermarkets,2382.60,56221.19,dec 2024.xlsx,Dec,2024,2024-12-01,12,FY 2024-25
1,5814,Fast food restaurants,1118.70,13664.52,dec 2024.xlsx,Dec,2024,2024-12-01,12,FY 2024-25
2,5812,Eating places and restaurants,992.78,16592.06,dec 2024.xlsx,Dec,2024,2024-12-01,12,FY 2024-25
3,4814,"Telecommunication services, including local an...",800.77,19110.23,dec 2024.xlsx,Dec,2024,2024-12-01,12,FY 2024-25
4,5541,Service stations (with or without ancillary se...,506.46,30717.58,dec 2024.xlsx,Dec,2024,2024-12-01,12,FY 2024-25


In [30]:
# Drop helper columns not needed for analysis and fix final column order
merch = merch[
    ["MCC", "Description","Volume_Mn", "Value_Cr", "Month", "Date", "Financial_Year"]
]

In [31]:
merch.head()

,MCC,Description,Volume_Mn,Value_Cr,Month,Date,Financial_Year
0,5411,Groceries and supermarkets,2382.60,56221.19,Dec,2024-12-01,FY 2024-25
1,5814,Fast food restaurants,1118.70,13664.52,Dec,2024-12-01,FY 2024-25
2,5812,Eating places and restaurants,992.78,16592.06,Dec,2024-12-01,FY 2024-25
3,4814,"Telecommunication services, including local an...",800.77,19110.23,Dec,2024-12-01,FY 2024-25
4,5541,Service stations (with or without ancillary se...,506.46,30717.58,Dec,2024-12-01,FY 2024-25


### 1.7 Export

In [36]:
merch.to_csv(merchant_category_dataf, index=False)

## 2. State-wise UPI Data
### 2.1 Load and combine all Excel files

In [11]:
state_files = glob.glob(os.path.join(STATE_FOLDER, "Ecosystem-Statistics-UPI-Upi-statewise-statistics-*.xlsx"))
print("Number of files:", len(state_files))

frames = []
for file in state_files:
    temp = pd.read_excel(file, header=1)                                          # header is on 2nd row
    temp = temp[["State / Union Territory", "Volume (in Mn)", "Value (in Cr.)"]].copy()
    temp.columns = ["State", "Volume_Mn", "Value_Cr"]                             # standard names
    temp["Source_File"] = os.path.basename(file)
    frames.append(temp)

state_df = pd.concat(frames, ignore_index=True)
print(state_df.shape)

Number of files: 37
(1369, 4)


### 2.2 Clean numeric columns

In [12]:
state_df["Volume_Mn"] = to_numeric_clean(state_df["Volume_Mn"])
state_df["Value_Cr"]  = to_numeric_clean(state_df["Value_Cr"])
print(state_df.isnull().sum())

State          0
Volume_Mn      0
Value_Cr       0
Source_File    0
dtype: int64


### 2.3 Extract Month, Date and Financial Year

In [13]:
# Month as 'Apr-2023' from the file name
ym = state_df["Source_File"].str.extract(rf"(20\d{{2}})-({MONTHS})")
state_df["Month"] = ym[1] + "-" + ym[0]

state_df["Date"] = pd.to_datetime("01-" + state_df["Month"], format="%d-%b-%Y")
state_df = add_financial_year(state_df)

state_df[["Source_File", "Month", "Date", "Financial_Year"]].head()

,Source_File,Month,Date,Financial_Year
0,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
1,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
2,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
3,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
4,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24


### 2.4 Clean State names

In [14]:
# Remove 'Unclassified' rows (not a real state)
state_df = state_df[
    ~state_df["State"].str.upper().str.replace(" ", "").str.startswith("UNCLASSIFIED")
]

# Standardise naming
state_df["State"] = state_df["State"].replace("ANDAMAN AND NICOBAR ISLANDS", "ANDAMAN & NICOBAR")
state_df["State"].value_counts()

State
MAHARASHTRA                           37
KARNATAKA                             37
TELANGANA                             37
UTTAR PRADESH                         37
TAMIL NADU                            37
MADHYA PRADESH                        37
DELHI                                 37
RAJASTHAN                             37
ANDHRA PRADESH                        37
GUJARAT                               37
BIHAR                                 37
WEST BENGAL                           37
HARYANA                               37
KERALA                                37
ODISHA                                37
PUNJAB                                37
JHARKHAND                             37
ASSAM                                 37
UTTARAKHAND                           37
CHHATTISGARH                          37
HIMACHAL PRADESH                      37
JAMMU AND KASHMIR                     37
GOA                                   37
CHANDIGARH                            37
PUDUCHERRY

### 2.5 Validate

In [15]:
print("Shape:", state_df.shape)
print("\nNull values:\n", state_df.isnull().sum())
print("\nDuplicates (State + Month):", state_df.duplicated(subset=["State", "Month"]).sum())
state_df.info()

Shape: (1332, 7)

Null values:
 State             0
Volume_Mn         0
Value_Cr          0
Source_File       0
Month             0
Date              0
Financial_Year    0
dtype: int64

Duplicates (State + Month): 36
<class 'pandas.core.frame.DataFrame'>
Index: 1332 entries, 0 to 1367
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   State           1332 non-null   object        
 1   Volume_Mn       1332 non-null   float64       
 2   Value_Cr        1332 non-null   float64       
 3   Source_File     1332 non-null   object        
 4   Month           1332 non-null   object        
 5   Date            1332 non-null   datetime64[ns]
 6   Financial_Year  1332 non-null   object        
dtypes: datetime64[ns](1), float64(2), object(4)
memory usage: 83.2+ KB


### 2.6 Export

In [20]:
state_df.head()

,State,Volume_Mn,Value_Cr,Source_File,Month,Date,Financial_Year
0,MAHARASHTRA,1859.62,258141.08,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
1,KARNATAKA,797.62,118039.69,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
2,TELANGANA,716.76,129532.28,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
3,UTTAR PRADESH,612.76,94729.13,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24
4,TAMIL NADU,459.82,78145.83,Ecosystem-Statistics-UPI-Upi-statewise-statist...,Apr-2023,2023-04-01,FY 2023-24


In [21]:
state_df.to_csv(STATE_OUT, index=False)

## 3. Monthly UPI Statistics
### 3.1 Load and combine all Excel files

In [22]:
monthly_files = glob.glob(os.path.join(MONTHLY_FOLDER, "*.xlsx"))
print("Number of files:", len(monthly_files))

frames = []
for file in monthly_files:
    temp = pd.read_excel(file)
    temp["Source_File"] = os.path.basename(file)
    frames.append(temp)

upi = pd.concat(frames, ignore_index=True)
print(upi.shape)
upi.head()

Number of files: 4
(41, 6)


,Month,Volume (In Mn.),Avg. Daily Volume (In Mn.),Value (In Cr.),Avg. Daily Value (In Cr.),Source_File
0,2024-03-01,13440.00,433.55,1978353.23,63817.85,Product-Statistics-UPI-Upi-monthly-statistics-...
1,2024-02-01,12102.67,417.33,1827869.35,63029.98,Product-Statistics-UPI-Upi-monthly-statistics-...
2,2024-01-01,12203.02,393.65,1841083.97,59389.81,Product-Statistics-UPI-Upi-monthly-statistics-...
3,2023-12-01,12020.23,387.75,1822949.45,58804.82,Product-Statistics-UPI-Upi-monthly-statistics-...
4,2023-11-01,11235.29,374.51,1739740.61,57991.35,Product-Statistics-UPI-Upi-monthly-statistics-...


### 3.2 Standardise column names

In [23]:
upi.columns = [
    "Month", "Volume_Mn", "Avg_Daily_Volume_Mn",
    "Value_Cr", "Avg_Daily_Value_Cr", "Source_File"
]

### 3.3 Create Date and Financial Year

In [24]:
upi["Date"] = pd.to_datetime(upi["Month"])
upi = add_financial_year(upi)

### 3.4 Clean numeric columns

In [25]:
for col in ["Volume_Mn", "Avg_Daily_Volume_Mn", "Value_Cr", "Avg_Daily_Value_Cr"]:
    upi[col] = to_numeric_clean(upi[col])

upi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Month                41 non-null     datetime64[ns]
 1   Volume_Mn            41 non-null     float64       
 2   Avg_Daily_Volume_Mn  41 non-null     float64       
 3   Value_Cr             41 non-null     float64       
 4   Avg_Daily_Value_Cr   41 non-null     float64       
 5   Source_File          41 non-null     object        
 6   Date                 41 non-null     datetime64[ns]
 7   Financial_Year       41 non-null     object        
dtypes: datetime64[ns](2), float64(4), object(2)
memory usage: 2.7+ KB


### 3.5 Final column order and validation

In [26]:
upi = upi[[
    "Month", "Date", "Financial_Year",
    "Volume_Mn", "Avg_Daily_Volume_Mn",
    "Value_Cr", "Avg_Daily_Value_Cr"
]]   # Source_File dropped, not needed for analysis

print("Null values:\n", upi.isnull().sum())
print("\nMonths per Financial Year:\n", upi["Financial_Year"].value_counts().sort_index())
upi.head()

Null values:
 Month                  0
Date                   0
Financial_Year         0
Volume_Mn              0
Avg_Daily_Volume_Mn    0
Value_Cr               0
Avg_Daily_Value_Cr     0
dtype: int64

Months per Financial Year:
 Financial_Year
FY 2023-24    12
FY 2024-25    12
FY 2025-26    12
FY 2026-27     5
Name: count, dtype: int64


,Month,Date,Financial_Year,Volume_Mn,Avg_Daily_Volume_Mn,Value_Cr,Avg_Daily_Value_Cr
0,2024-03-01,2024-03-01,FY 2023-24,13440.00,433.55,1978353.23,63817.85
1,2024-02-01,2024-02-01,FY 2023-24,12102.67,417.33,1827869.35,63029.98
2,2024-01-01,2024-01-01,FY 2023-24,12203.02,393.65,1841083.97,59389.81
3,2023-12-01,2023-12-01,FY 2023-24,12020.23,387.75,1822949.45,58804.82
4,2023-11-01,2023-11-01,FY 2023-24,11235.29,374.51,1739740.61,57991.35


### 3.6 Export

In [27]:
upi.to_csv(MONTHLY_OUT, index=False)